# Notebook 02 — Train YOLOv8 Sign Detector
✔ YOLO training requirement

## Cell 1 — Imports

In [2]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
from PIL import Image
import os
print("Ultralytics ready")

Ultralytics ready


## Cell 2 — Train YOLOv8
This one cell handles the entire YOLOv8 training loop. Loss curves are saved automatically.

In [3]:
# Load YOLOv8 small — pretrained on COCO dataset
model = YOLO("yolov8s.pt")

print("Starting YOLOv8 training...")
print("This will take 10-30 minutes depending on your hardware.")
print("Watch box_loss and cls_loss — they should decrease each epoch.\n")

results = model.train(
    data    = "data_yolo/data.yaml",
    epochs  = 10,
    imgsz   = 640,
    batch   = 16,
    name    = "campus_signs",
    patience= 10,          # early stopping if no improvement
    verbose = True
)

print("\n✓ Training complete!")
print("Best model saved to: runs/detect/campus_signs/weights/best.pt")
print("Loss curves saved to: runs/detect/campus_signs/results.png")

Starting YOLOv8 training...
This will take 10-30 minutes depending on your hardware.
Watch box_loss and cls_loss — they should decrease each epoch.

Ultralytics 8.4.33  Python-3.14.0 torch-2.11.0+cpu CPU (Intel Core Ultra 7 155H)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data_yolo/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, 

## Cell 3 — Show YOLO loss curves (for submission)

In [4]:
# Display the automatically saved loss curves
results_img = Image.open("runs/detect/campus_signs/labels.jpg")
plt.figure(figsize=(14, 6))
plt.imshow(results_img)
plt.axis("off")
plt.title("YOLOv8 Training Results — Label Distribution", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("yolo_loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved as yolo_loss_curves.png — include this in your report")

<Figure size 1400x600 with 1 Axes>

✓ Saved as yolo_loss_curves.png — include this in your report


## Cell 4 — Visual Prediction with YOLOv8 (for submission)
✔ Visual predictions requirement

In [5]:
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from ultralytics import YOLO
import os

# Load best trained model
best_model = YOLO("runs/detect/campus_signs/weights/best.pt")

# Get a test image — picks first image from val set
val_images = os.listdir("data_yolo/images/val")
test_img_path = f"data_yolo/images/val/{val_images[3]}"

print(f"Running YOLOv8 on: {test_img_path}")

# Run detection
results = best_model(test_img_path, conf=0.3, verbose=False)

# Show annotated image with bounding boxes
annotated = results[0].plot()  # draws boxes on image
plt.figure(figsize=(12, 8))
plt.imshow(annotated[:, :, ::-1])  # BGR → RGB
plt.title("YOLOv8 — Detected Signs (Visual Prediction)", fontsize=14, fontweight="bold")
plt.axis("off")
plt.tight_layout()
plt.savefig("yolo_visual_prediction.png", dpi=150, bbox_inches="tight")
plt.show()

# Print what was detected
print("\nDetections:")
class_names = {0: "building_sign", 1: "room_number"}
if len(results[0].boxes) == 0:
    print("  No signs detected in this image (try another image)")
else:
    for box in results[0].boxes:
        cls  = int(box.cls[0])
        conf = float(box.conf[0])
        print(f"  ✓ {class_names.get(cls, cls)}  —  confidence {conf:.1%}")

print("\n✓ Saved as yolo_visual_prediction.png")

Running YOLOv8 on: data_yolo/images/val/IMG20260323150338.jpg


<Figure size 1200x800 with 1 Axes>


Detections:
  ✓ room_number  —  confidence 59.3%

✓ Saved as yolo_visual_prediction.png


## Cell 5 — Test on multiple images

In [6]:
# Run on first 4 val images and show in a grid
val_images = os.listdir("data_yolo/images/val")[7:11]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, img_name in enumerate(val_images):
    path = f"data_yolo/images/val/{img_name}"
    res  = best_model(path, conf=0.3, verbose=False)
    ann  = res[0].plot()
    axes[i].imshow(ann[:, :, ::-1])
    n_det = len(res[0].boxes)
    axes[i].set_title(f"{img_name[:20]}...  ({n_det} detections)",
                      fontsize=10)
    axes[i].axis("off")

plt.suptitle("YOLOv8 — Visual Predictions on Val Set",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("yolo_grid_predictions.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved as yolo_grid_predictions.png")

<Figure size 1400x1000 with 4 Axes>

✓ Saved as yolo_grid_predictions.png
